# 实验三：PyBaMM 电池仿真

## 实验目标
- 理解锂电池 DFN/SPM 等效电路模型
- 仿真恒流充放电，看电压-SOC 曲线
- 对比不同倍率放电对可用容量的影响

## 背景
PyBaMM 是开源电池仿真工具，被特斯拉/牛津等用于电池研发。本实验用单颗粒模型（SPM）跑充放电。

In [ ]:
import pybamm
pybamm.set_logging_level('ERROR')
print('PyBaMM ok')
import matplotlib.pyplot as plt
import numpy as np

## 1. 1C 恒流放电

In [ ]:
model = pybamm.lithium_ion.SPM()
param = pybamm.ParameterValues('Chen2020')
sim = pybamm.Simulation(model, parameter_values=param)

# 1C 放电到 2.5V 截止
sim.solve([0, 3600])
sol = sim.solution

t = sol['Time [s]'].entries
V = sol['Terminal voltage [V]'].entries

plt.figure(figsize=(10, 4))
plt.plot(t/60, V)
plt.xlabel('时间 (min)'); plt.ylabel('端电压 (V)')
plt.title('SPM 模型 1C 恒流放电曲线')
plt.grid(True, alpha=0.3); plt.show()

print('放电时长: %.1f min' % (t[-1]/60))

## 2. 不同倍率对比
0.5C / 1C / 2C 放电，看倍率对放出容量的影响（极化内阻效应）。

In [ ]:
plt.figure(figsize=(10, 4))
for crate in [0.5, 1, 2]:
    # 用 Experiment 接口设倍率
    exp = pybamm.Experiment([f'Discharge at {crate}C until 2.5V'])
    s = pybamm.Simulation(model, parameter_values=param, experiment=exp)
    s.solve()
    sol = s.solution
    t = sol['Time [s]'].entries
    V = sol['Terminal voltage [V]'].entries
    cap = sol['Discharge capacity [A.h]'].entries
    plt.plot(cap, V, label=f'{crate}C')
    print(f'{crate}C: 放出容量 {cap[-1]:.2f} Ah')

plt.xlabel('放出容量 (Ah)'); plt.ylabel('端电压 (V)')
plt.title('不同倍率放电曲线对比')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

## 思考题
1. 为什么 2C 放电的可用容量比 0.5C 少？（提示：极化）
2. 如果把截止电压从 2.5V 提高到 3.0V，对电池寿命有什么好处？代价是什么？
3. 试着换 `pybamm.lithium_ion.DFN()`（更精确但更慢），对比曲线差异。